In [1]:
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [2]:
input_path = Path("../data/processed/products_features.csv")

if not input_path.exists():
    raise FileNotFoundError("Exécute d'abord 03_feature_engineering.ipynb.")

df = pd.read_csv(input_path, encoding="utf-8-sig")
print("Dimensions :", df.shape)
display(df.head())


Dimensions : (9, 23)


,product_id,name,domain,category,subcategory,brand,model,price_dzd,availability,feature_text,...,pressure_rating_bar,power_kw,voltage_v,price_dzd.1,diameter_mm_missing,length_m_missing,pressure_rating_bar_missing,power_kw_missing,voltage_v_missing,price_dzd_missing
0,SH001,chauffe-bain 11l junkers,chauffage,chaudière / générateur,chauffe-bain gaz,junkers,NaN,28500.0,In stock,chauffe-bain 11l junkers chauffage chaudière /...,...,NaN,NaN,NaN,28500.0,1,1,1,1,1,0
1,SH002,chauffe-eau smart 12l energical,chauffage,chaudière / générateur,chauffe-eau gaz,energical,smart 12l,29500.0,In stock,chauffe-eau smart 12l energical chauffage chau...,...,NaN,NaN,NaN,29500.0,1,1,1,1,1,0
2,SH003,chaudière ventouse 28 kw energical,chauffage,chaudière / générateur,chaudière gaz,energical,28 kw ventouse,NaN,NaN,chaudière ventouse 28 kw energical chauffage c...,...,NaN,28.0,NaN,NaN,1,1,1,0,1,1
3,SH004,accélérateur grundfos 15-50 75w réf. 99062433,chauffage,pompe de circulation,circulateur / accélérateur chaudière,grundfos,15-50,21800.0,NaN,accélérateur grundfos 15-50 75w réf. 99062433 ...,...,NaN,NaN,NaN,21800.0,1,1,1,1,1,0
4,SH005,accélérateur grundfos 15-60 110w réf. 99157090,chauffage,pompe de circulation,circulateur / accélérateur chaudière,grundfos,15-60,21300.0,NaN,accélérateur grundfos 15-60 110w réf. 99157090...,...,NaN,NaN,NaN,21300.0,1,1,1,1,1,0


In [3]:
required_columns = ["product_id", "name", "feature_text"]
missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Colonnes nécessaires absentes : {missing_columns}")

df = df.dropna(subset=["product_id"]).copy()
df["product_id"] = df["product_id"].astype(str)
df["feature_text"] = df["feature_text"].fillna("").astype(str)

print("Produits utilisables :", len(df))
print("Textes vides :", df["feature_text"].str.strip().eq("").sum())


Produits utilisables : 9
Textes vides : 0


In [4]:
df["feature_text"] = df["feature_text"].str.strip()
df_model = df[df["feature_text"] != ""].copy()

if df_model.empty:
    raise ValueError("Aucun produit ne possède de feature_text exploitable.")

df_model = df_model.reset_index(drop=True)
print("Produits avec texte :", len(df_model))
display(df_model[["product_id", "name", "feature_text"]].head())


Produits avec texte : 9


,product_id,name,feature_text
0,SH001,chauffe-bain 11l junkers,chauffe-bain 11l junkers chauffage chaudière /...
1,SH002,chauffe-eau smart 12l energical,chauffe-eau smart 12l energical chauffage chau...
2,SH003,chaudière ventouse 28 kw energical,chaudière ventouse 28 kw energical chauffage c...
3,SH004,accélérateur grundfos 15-50 75w réf. 99062433,accélérateur grundfos 15-50 75w réf. 99062433 ...
4,SH005,accélérateur grundfos 15-60 110w réf. 99157090,accélérateur grundfos 15-60 110w réf. 99157090...


In [5]:
tfidf = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=1
)

tfidf_matrix = tfidf.fit_transform(df_model["feature_text"])

print("Dimensions de la matrice TF-IDF :", tfidf_matrix.shape)
print("Nombre de termes du vocabulaire :", len(tfidf.get_feature_names_out()))


Dimensions de la matrice TF-IDF : (9, 251)
Nombre de termes du vocabulaire : 251


In [6]:
print("Nombre de produits :", tfidf_matrix.shape[0])
print("Nombre de caractéristiques textuelles :", tfidf_matrix.shape[1])

# Afficher quelques termes et poids du premier produit
first_vector = tfidf_matrix[0].toarray().ravel()
top_positions = first_vector.argsort()[::-1][:10]
terms = tfidf.get_feature_names_out()

pd.DataFrame({
    "terme": terms[top_positions],
    "poids_tfidf": first_vector[top_positions]
})


Nombre de produits : 9
Nombre de caractéristiques textuelles : 251


,terme,poids_tfidf
0,chauffe bain,0.363960
1,bain,0.363960
2,chauffe,0.307407
3,junkers,0.242640
4,la,0.178187
5,supplémentaires vérifiés,0.121320
6,sans,0.121320
7,nom,0.121320
8,supplémentaires,0.121320
9,sans détails,0.121320


In [7]:
similarity_matrix = cosine_similarity(tfidf_matrix)

print("Dimensions de la matrice de similarité :", similarity_matrix.shape)
display(pd.DataFrame(
    similarity_matrix,
    index=df_model["product_id"],
    columns=df_model["product_id"]
).round(3))


Dimensions de la matrice de similarité : (9, 9)


product_id,SH001,SH002,SH003,SH004,SH005,SH006,SH007,SH008,SH009
product_id,,,,,,,,,
SH001,1.000,0.155,0.049,0.007,0.007,0.007,0.029,0.033,0.000
SH002,0.155,1.000,0.107,0.011,0.011,0.011,0.010,0.014,0.006
SH003,0.049,0.107,1.000,0.028,0.028,0.028,0.057,0.050,0.006
SH004,0.007,0.011,0.028,1.000,0.439,0.439,0.108,0.118,0.008
SH005,0.007,0.011,0.028,0.439,1.000,0.439,0.108,0.118,0.008
SH006,0.007,0.011,0.028,0.439,0.439,1.000,0.108,0.118,0.008
SH007,0.029,0.010,0.057,0.108,0.108,0.108,1.000,0.516,0.006
SH008,0.033,0.014,0.050,0.118,0.118,0.118,0.516,1.000,0.006
SH009,0.000,0.006,0.006,0.008,0.008,0.008,0.006,0.006,1.000


In [8]:
def recommend_similar_products(product_id, top_n=5):
    product_id = str(product_id)

    matches = df_model.index[df_model["product_id"] == product_id].tolist()
    if not matches:
        raise ValueError(f"Identifiant produit introuvable ou sans texte : {product_id}")

    product_index = matches[0]
    scores = list(enumerate(similarity_matrix[product_index]))

    # Exclure le produit lui-même
    scores = [(idx, score) for idx, score in scores if idx != product_index]
    scores.sort(key=lambda item: item[1], reverse=True)

    top_scores = scores[:top_n]
    result_indices = [idx for idx, score in top_scores]

    result = df_model.iloc[result_indices][
        [col for col in ["product_id", "name", "domain", "category", "brand", "price_dzd"]
         if col in df_model.columns]
    ].copy()

    result["similarity_score"] = [score for idx, score in top_scores]
    return result.reset_index(drop=True)


In [9]:
# Afficher les identifiants disponibles pour choisir un produit
display(df_model[["product_id", "name"]])

# Exemple : remplacer SH003 par un identifiant présent dans le tableau
example_product_id = df_model.iloc[0]["product_id"]
print("Produit choisi :", example_product_id)

recommendations = recommend_similar_products(example_product_id, top_n=5)
display(recommendations)


,product_id,name
0,SH001,chauffe-bain 11l junkers
1,SH002,chauffe-eau smart 12l energical
2,SH003,chaudière ventouse 28 kw energical
3,SH004,accélérateur grundfos 15-50 75w réf. 99062433
4,SH005,accélérateur grundfos 15-60 110w réf. 99157090
5,SH006,accélérateur grundfos 15-65 120w réf. 99062424
6,SH007,accélérateur wilo 82w avec bloc pour chaudière...
7,SH008,accélérateur wilo 82w pour chaudière saunier d...
8,SH009,adaptateur de robinet réglable


Produit choisi : SH001


,product_id,name,domain,category,brand,price_dzd,similarity_score
0,SH002,chauffe-eau smart 12l energical,chauffage,chaudière / générateur,energical,29500.0,0.154967
1,SH003,chaudière ventouse 28 kw energical,chauffage,chaudière / générateur,energical,NaN,0.048764
2,SH008,accélérateur wilo 82w pour chaudière saunier d...,chauffage,pompe de circulation,wilo,24000.0,0.033456
3,SH007,accélérateur wilo 82w avec bloc pour chaudière...,chauffage,pompe de circulation,wilo,21300.0,0.029217
4,SH004,accélérateur grundfos 15-50 75w réf. 99062433,chauffage,pompe de circulation,grundfos,21800.0,0.006950


In [10]:
def find_products(keyword):
    keyword = str(keyword).strip().lower()
    mask = df_model["name"].fillna("").str.lower().str.contains(keyword, regex=False)
    return df_model.loc[mask, ["product_id", "name"]].head(20)

display(find_products("chaudière"))


,product_id,name
2,SH003,chaudière ventouse 28 kw energical
6,SH007,accélérateur wilo 82w avec bloc pour chaudière...
7,SH008,accélérateur wilo 82w pour chaudière saunier d...


In [11]:
output_path = Path("../data/processed/content_recommendations.csv")
all_results = []

for product_id in df_model["product_id"]:
    try:
        recs = recommend_similar_products(product_id, top_n=5)
        recs.insert(0, "query_product_id", product_id)
        all_results.append(recs)
    except ValueError:
        continue

if all_results:
    recommendations_df = pd.concat(all_results, ignore_index=True)
else:
    recommendations_df = pd.DataFrame()

recommendations_df.to_csv(output_path, index=False, encoding="utf-8-sig")
print("Fichier créé :", output_path.exists())
print("Nombre de recommandations exportées :", len(recommendations_df))


Fichier créé : True
Nombre de recommandations exportées : 45


In [12]:
if not recommendations_df.empty:
    display(
        recommendations_df.sort_values(
            ["query_product_id", "similarity_score"],
            ascending=[True, False]
        ).head(30)
    )

print("Rappel : vérifier les fiches techniques avant toute conclusion de compatibilité.")


,query_product_id,product_id,name,domain,category,brand,price_dzd,similarity_score
0,SH001,SH002,chauffe-eau smart 12l energical,chauffage,chaudière / générateur,energical,29500.0,0.154967
1,SH001,SH003,chaudière ventouse 28 kw energical,chauffage,chaudière / générateur,energical,NaN,0.048764
2,SH001,SH008,accélérateur wilo 82w pour chaudière saunier d...,chauffage,pompe de circulation,wilo,24000.0,0.033456
3,SH001,SH007,accélérateur wilo 82w avec bloc pour chaudière...,chauffage,pompe de circulation,wilo,21300.0,0.029217
4,SH001,SH004,accélérateur grundfos 15-50 75w réf. 99062433,chauffage,pompe de circulation,grundfos,21800.0,0.006950
5,SH002,SH001,chauffe-bain 11l junkers,chauffage,chaudière / générateur,junkers,28500.0,0.154967
6,SH002,SH003,chaudière ventouse 28 kw energical,chauffage,chaudière / générateur,energical,NaN,0.107114
7,SH002,SH008,accélérateur wilo 82w pour chaudière saunier d...,chauffage,pompe de circulation,wilo,24000.0,0.013753
8,SH002,SH004,accélérateur grundfos 15-50 75w réf. 99062433,chauffage,pompe de circulation,grundfos,21800.0,0.010592
9,SH002,SH005,accélérateur grundfos 15-60 110w réf. 99157090,chauffage,pompe de circulation,grundfos,21300.0,0.010592


Rappel : vérifier les fiches techniques avant toute conclusion de compatibilité.
